# Lab de Limpeza - Camada Silver: Clientes

**Comentário:** Descreva aqui as regras de negócio identificadas durante a limpeza manual.

In [ ]:
import pandas as pd
import s3fs

storage_options = {
    "key": "minio",
    "secret": "minio123",
    "client_kwargs": {"endpoint_url": "http://localhost:9000"}
}

fs = s3fs.S3FileSystem(**storage_options)
arquivos = fs.glob("bronze/clientes/**/*.csv")

df = pd.concat([pd.read_csv(f"s3://{'{arquivo}'}", storage_options=storage_options) for arquivo in arquivos], ignore_index=True)
df.head()

In [8]:
cols_str = ["nome_cliente", "canal", "regiao", "cidade", "uf", "representante"]

for col in cols_str:
    df[col] = df[col].astype(str).str.strip()

In [9]:
df["_ingested_at"] = pd.to_datetime(df["_ingested_at"], errors="coerce")

In [10]:
df["uf"] = df["uf"].str.upper()

In [11]:
df["ativo"].unique()

array(['S', '1', ' S', 'Sim', 'SIM', 's'], dtype=object)

In [12]:
def normalize_ativo(value):
    if pd.isna(value):
        return None

    v = str(value).strip().lower()

    true_values = {"s", "sim", "1", "true", "t", "yes"}
    false_values = {"n", "nao", "não", "0", "false", "f", "no"}

    if v in true_values:
        return True
    elif v in false_values:
        return False
    else:
        return None 


df["ativo"] = df["ativo"].apply(normalize_ativo)

In [13]:
df = df[df["cnpj"].notna()]
df = df[df["nome_cliente"].notna()]

In [15]:
df = df.sort_values("_ingested_at")

In [16]:
df = df.drop_duplicates(
    subset=["cnpj"],
    keep="last"
)

In [ ]:
df.to_parquet(
    "data/tmp/clientes.parquet",
    engine="pyarrow",
    index=False
)

OSError: Cannot save file into a non-existent directory: 'data/temp'